# 3. Actual energy and climate results

Run the baseline, both changes together, and the two renovation steps with the current RC engine. This is an actual annual calculation using the bundled Gothenburg weather. It is a teaching comparison, not a comparison with measured energy.

RC settings explicitly select its versioned model, omit supply-temperature limits and latent recovery, and approximate ground transfer with a conductance multiplier. These are separate from the LCA period and climate factors. The shared helper retains the full choices in the energy diagnostics.

In [ ]:
from pathlib import Path

from platform_walkthrough import run_case
from renovation_case import rc_settings

print("RC modelling choices:", rc_settings().effective_configuration())
result = run_case(output=Path("../outputs/notebooks/actual-rc"))
print("Saved results:", result["directory"].resolve())

Annual purchased energy is reported by carrier. Each saved evaluation supplies its own energy to LCA through annual_energy(). A later edit to a building cannot relabel the old energy as describing the new building. Incomplete or non-annual energy is rejected.

In [ ]:
for name, evaluation in result["evaluations"].items():
    print(name, evaluation.energy.delivered_by_carrier)
    print(evaluation.annual_energy().source)
    assert evaluation.lca.complete

for row in result["staged"].table():
    print(row)

Material impact so far is accumulated renovation work. Energy impact is this state’s annual carrier energy multiplied by 30 years and the stated carrier factors. Total impact adds those two amounts. Savings subtract this total from the baseline total, so positive values mean a reduction. Each row uses the same 30-year comparison period; energy from different rows is never added together.

Carrier factors (0.05 kg CO₂e/kWh district heat; 0.1 electricity), heating efficiency and window-removal factors are teaching assumptions. They are not Boverket data. Future replacement, final demolition and dated operating periods are outside this calculation.

In [ ]:
from math import isclose

single = result["single"]
staged = result["staged"]
assert isclose(
    single.stages[-1].total_kgco2e, staged.stages[-1].total_kgco2e, rel_tol=1e-10
)
for row, name in zip(staged.table(), ("baseline", "step_1", "step_2")):
    assert isclose(
        row["total_kgco2e"],
        result["evaluations"][name].lca.total_gwp_kgco2e,
        rel_tol=1e-10,
    )
print("Same final answer for single and staged work:", staged.stages[-1].total_kgco2e)
figure = staged.plot()
figure

The command-line walkthrough also supports EnergyPlus with an explicitly installed executable, and an optional Workflows search. See the example index for the commands. RC and EnergyPlus use different numerical methods; changes to RC recovery, ground and gain handling mean earlier teaching outputs should be recalculated.